LoRA r=8, simple prompt, loss-based inference

Back to attempt8's proven recipe (best single-model result so far),
reduced to meet the **≤ 5M trainable parameter** competition constraint.

- Simple prompt: `lecture` + `hint` context only — no metadata, no captions
- Standard LoRA r=8, lora_alpha=16, no DoRA
- Cache reused: `train_cache.pt` (same prompt format as attempt8)
- Inference: pure loss-based scoring
- Submission: single-model (ep5) + weight soup (ep4+ep5)

In [ ]:
import sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_DIR   = Path('/content/drive/MyDrive/DL/endterm/pixels-to-predictions')
    CKPT_DIR   = Path('/content/drive/MyDrive/DL/endterm/checkpoints15')
    CACHE_PATH = Path('/content/drive/MyDrive/DL/endterm/train_cache.pt')
else:
    DATA_DIR   = Path('pixels-to-predictions')
    CKPT_DIR   = Path('checkpoints15')
    CACHE_PATH = Path('train_cache.pt')

CKPT_DIR.mkdir(parents=True, exist_ok=True)
print(f"{'Colab' if IN_COLAB else 'Local'} | DATA_DIR: {DATA_DIR} | CKPT_DIR: {CKPT_DIR}")
print(f"Shared cache: {CACHE_PATH}")

Mounted at /content/drive
Colab | DATA_DIR: /content/drive/MyDrive/DL/endterm/pixels-to-predictions | CKPT_DIR: /content/drive/MyDrive/DL/endterm/checkpoints15
Shared cache: /content/drive/MyDrive/DL/endterm/train_cache.pt


In [ ]:
%pip install -q "https://github.com/lesj0610/flash-attention/releases/download/v2.8.3-cu12-torch2.10-cp312/flash_attn-2.8.3%2Bcu12torch2.10cxx11abiTRUE-cp312-cp312-linux_x86_64.whl"
%pip install -q datasets trl transformers accelerate peft bitsandbytes pandas lxml
%pip install -q "torchao>=0.16.0"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.6/253.6 MB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 721.6/721.6 kB 33.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 71.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 74.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 90.4 MB/s eta 0:00:00


In [ ]:
import os, json, random
import numpy as np
import pandas as pd
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader

SEED = 51
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

MODEL_ID   = "HuggingFaceTB/SmolVLM-500M-Instruct"
IMG_SIZE   = 336
MAX_EPOCHS = 5
LR         = 2e-4

if torch.cuda.is_available():
    DEVICE = torch.device('cuda')
    DTYPE  = torch.bfloat16
elif torch.backends.mps.is_available():
    DEVICE = torch.device('mps')
    DTYPE  = torch.float16
else:
    DEVICE = torch.device('cpu')
    DTYPE  = torch.float32

SMOKE_TEST = (DEVICE.type != 'cuda')

print(f"Device: {DEVICE} | dtype: {DTYPE} | SEED: {SEED} | Smoke: {SMOKE_TEST}")

Device: cuda | dtype: torch.bfloat16 | SEED: 51 | Smoke: False


In [ ]:
# ── Load CSVs ─────────────────────────────────────────────────────────────────
train_df = pd.read_csv(DATA_DIR / 'train.csv')
val_df   = pd.read_csv(DATA_DIR / 'val.csv')
test_df  = pd.read_csv(DATA_DIR / 'test.csv')

for df in [train_df, val_df, test_df]:
    df['choices'] = df['choices'].apply(json.loads)

mini_val_df   = val_df.sample(min(200, len(val_df)), random_state=SEED).reset_index(drop=True)
_n_all_images = len(train_df) + len(val_df) + len(test_df)

train_df = pd.concat([train_df, val_df], ignore_index=True)

if SMOKE_TEST:
    train_df    = train_df.sample(48, random_state=SEED).reset_index(drop=True)
    mini_val_df = mini_val_df.sample(16, random_state=SEED).reset_index(drop=True)
    test_df     = test_df.sample(16, random_state=SEED).reset_index(drop=True)

print(f"Train (train+val): {len(train_df)} | Mini-val: {len(mini_val_df)} | Test: {len(test_df)}")

# ── Path auto-fix ──────────────────────────────────────────────────────────────
sample_path = DATA_DIR / train_df.iloc[0]['image_path']
if not sample_path.exists():
    alt = DATA_DIR / 'images' / train_df.iloc[0]['image_path']
    if alt.exists():
        print("Extra images/ nesting detected — adjusting DATA_DIR")
        DATA_DIR = DATA_DIR / 'images'
    else:
        print(f"WARNING: image not found at {sample_path} — check DATA_DIR")
else:
    print(f"Paths OK: {sample_path}")

# ── Copy images to local SSD (Colab only) ─────────────────────────────────────
if IN_COLAB:
    import shutil
    _local_img_dir = Path('/content/images')
    _n_existing    = len(list(_local_img_dir.rglob('*.png'))) if _local_img_dir.exists() else 0
    if _n_existing < _n_all_images:
        if _local_img_dir.exists():
            shutil.rmtree(_local_img_dir)
        print(f"Copying {_n_all_images} images to local SSD …")
        shutil.copytree(DATA_DIR / 'images', _local_img_dir)
        print(f"Done — {len(list(_local_img_dir.rglob('*.png')))} images")
    else:
        print(f"Local images already present ({_n_existing} files)")
    IMG_DATA_DIR = Path('/content')
else:
    IMG_DATA_DIR = DATA_DIR

print(f"Image reads → {IMG_DATA_DIR}")

Train (train+val): 4157 | Mini-val: 200 | Test: 1008
Extra images/ nesting detected — adjusting DATA_DIR
Copying 5165 images to local SSD …
Done — 5165 images
Image reads → /content


In [ ]:
CHOICE_LETTERS = "ABCDEFGHIJ"

def build_prompt(row, include_answer=False):
    parts = []
    for field in ('lecture', 'hint'):
        v = row.get(field, '')
        if pd.notna(v) and str(v).strip():
            parts.append(str(v).strip())
    ctx = "\n".join(parts)
    choices_str = "\n".join(
        f"  {CHOICE_LETTERS[i]}. {c}" for i, c in enumerate(row['choices'])
    )
    prompt = "<image>\n"
    if ctx:
        prompt += f"Context:\n{ctx}\n\n"
    prompt += f"Question: {row['question']}\nChoices:\n{choices_str}\nAnswer:"
    if include_answer:
        prompt += f" {CHOICE_LETTERS[int(row['answer'])]}"
    return prompt


class ScienceQADataset(Dataset):
    def __init__(self, df, img_dir, img_size=336):
        self.df       = df.reset_index(drop=True)
        self.img_dir  = img_dir
        self.img_size = img_size

    def __len__(self):
        return len(self.df)

    def _load(self, rel_path):
        return Image.open(self.img_dir / rel_path).convert('RGB').resize(
            (self.img_size, self.img_size), Image.BICUBIC
        )

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = self._load(row['image_path'])
        return {
            'image':   img,
            'text':    build_prompt(row, include_answer=False),
            'choices': row['choices'],
            'id':      row['id'],
            'answer':  int(row['answer']) if 'answer' in row.index else -1,
        }


mini_val_ds = ScienceQADataset(mini_val_df, IMG_DATA_DIR, IMG_SIZE)
test_ds     = ScienceQADataset(test_df,     IMG_DATA_DIR, IMG_SIZE)
print(f"mini_val={len(mini_val_ds)} | test={len(test_ds)}")
print(build_prompt(train_df.iloc[0], include_answer=True))

mini_val=200 | test=1008
<image>
Context:
Animals increase their reproductive success when they have offspring that survive to reproduce.
Animals can increase their chances of having offspring by behaving in ways that help them get partners to mate and reproduce with. These partners are called mates. For example, animals may make special sounds, perform specific dances, or show off bright colors to attract mates. Animals may also compete with each other for mates.
Animals can increase the chances that their offspring will survive to reproduce by caring for and protecting them. For example, animals may feed their offspring or guard them from predators. These behaviors increase the chances that the offspring will survive to adulthood, when they can reproduce.
Many behaviors can increase the chances that animals will have offspring that survive to reproduce. But the behaviors cannot guarantee that the animals will have greater reproductive success. Animals that attract or compete for mate

In [ ]:
from transformers import AutoProcessor, AutoModelForImageTextToText
from peft import LoraConfig, get_peft_model, TaskType

processor = AutoProcessor.from_pretrained(MODEL_ID)
if processor.tokenizer.pad_token is None:
    processor.tokenizer.pad_token = processor.tokenizer.eos_token

load_kwargs = dict(
    dtype=DTYPE,
    low_cpu_mem_usage=True,
    device_map="auto" if DEVICE.type == "cuda" else None,
)
try:
    model = AutoModelForImageTextToText.from_pretrained(
        MODEL_ID, attn_implementation="flash_attention_2", **load_kwargs
    )
    print("FlashAttention-2 enabled")
except Exception as e:
    model = AutoModelForImageTextToText.from_pretrained(MODEL_ID, **load_kwargs)
    print(f"Eager attention ({type(e).__name__})")

if DEVICE.type != "cuda":
    model.to(DEVICE)

lora_cfg = LoraConfig(
    r=8,
    lora_alpha=16,
    use_dora=False,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj", "out_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)
model = get_peft_model(model, lora_cfg)
model.config.use_cache = False
model.gradient_checkpointing_enable()
model.print_trainable_parameters()

n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
if n_trainable > 5_000_000:
    raise ValueError(
        f"Trainable params {n_trainable:,} ({n_trainable/1e6:.2f}M) exceeds 5M cap. "
        f"Reduce r (currently 8) to 6 and re-run this cell."
    )
print(f"Trainable params: {n_trainable:,} ({n_trainable/1e6:.2f}M) — within 5M cap")
model.eval()
print("Model ready (LoRA r=8).")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


processor_config.json:   0%|          | 0.00/68.0 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/429 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/486 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

special_tokens_map.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.02G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/489 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/136 [00:00<?, ?B/s]

FlashAttention-2 enabled
trainable params: 4,931,584 || all params: 512,413,888 || trainable%: 0.9624
Trainable params: 4,931,584 (4.93M) — within 5M cap
Model ready (LoRA r=8).


In [ ]:
# ── Pre-tokenize training set (text tokens only, cached to Drive) ──────────────
import torch.nn.functional as F
from tqdm.auto import tqdm

processor.tokenizer.padding_side = 'right'
# CACHE_PATH set in cell-02 — shared across all attempts (same data, same model, deterministic)

def _tokenize_row(row):
    prompt = build_prompt(row, include_answer=True)
    img    = Image.open(IMG_DATA_DIR / row['image_path']).convert('RGB').resize(
                 (IMG_SIZE, IMG_SIZE), Image.BICUBIC)
    enc    = processor(text=[prompt], images=[img], return_tensors='pt')
    letter = prompt.rsplit(' ', 1)[1]
    n_ans  = len(processor.tokenizer.encode(f" {letter}", add_special_tokens=False))
    return {
        'input_ids':      enc['input_ids'].squeeze(0),
        'attention_mask': enc['attention_mask'].squeeze(0),
        'n_ans':          n_ans,
        'image_path':     row['image_path'],
    }

if CACHE_PATH.exists():
    train_cache = torch.load(CACHE_PATH, map_location='cpu', weights_only=False)
    if len(train_cache) == len(train_df):
        print(f"Loaded cache: {len(train_cache)} samples ({CACHE_PATH.stat().st_size/1e6:.1f} MB)")
    else:
        print(f"Partial cache ({len(train_cache)}/{len(train_df)}) — resuming …")
else:
    train_cache = []
    print("No cache — tokenizing from scratch …")

if len(train_cache) < len(train_df):
    start = len(train_cache)
    for _, row in tqdm(train_df.iloc[start:].iterrows(), total=len(train_df) - start):
        train_cache.append(_tokenize_row(row))
    torch.save(train_cache, CACHE_PATH)
    print(f"Saved {len(train_cache)} samples → {CACHE_PATH} ({CACHE_PATH.stat().st_size/1e6:.1f} MB)")

N_TRAIN    = len(train_cache)
TRAIN_BS   = 8
GRAD_ACCUM = 4

def collate_train(batch):
    max_len = max(b['input_ids'].shape[0] for b in batch)
    out = {}
    for k in ('input_ids', 'attention_mask'):
        pad_val = processor.tokenizer.pad_token_id if k == 'input_ids' else 0
        out[k] = torch.stack([
            F.pad(b[k], (0, max_len - b[k].shape[0]), value=pad_val) for b in batch
        ])
    imgs = [
        Image.open(IMG_DATA_DIR / b['image_path']).convert('RGB').resize(
            (IMG_SIZE, IMG_SIZE), Image.BICUBIC)
        for b in batch
    ]
    img_enc = processor.image_processor(images=imgs, return_tensors='pt')
    out['pixel_values'] = img_enc['pixel_values']
    if 'pixel_attention_mask' in img_enc:
        out['pixel_attention_mask'] = img_enc['pixel_attention_mask']
    labels = torch.full_like(out['input_ids'], -100)
    for i, b in enumerate(batch):
        ul, n = b['input_ids'].shape[0], b['n_ans']
        labels[i, ul - n : ul] = out['input_ids'][i, ul - n : ul]
    out['labels'] = labels
    return out

print(f"\nReady — {N_TRAIN} samples | TRAIN_BS={TRAIN_BS} | GRAD_ACCUM={GRAD_ACCUM} | effective_BS={TRAIN_BS*GRAD_ACCUM}")

Loaded cache: 4157 samples (93.8 MB)

Ready — 4157 samples | TRAIN_BS=8 | GRAD_ACCUM=4 | effective_BS=32


In [ ]:
# ── Fine-tuning with checkpoint save / resume ──────────────────────────────────
import json
import warnings
import safetensors.torch as sf
from torch.optim import AdamW
from transformers import get_cosine_schedule_with_warmup
from peft import set_peft_model_state_dict
from tqdm.auto import tqdm

def save_ckpt(tag, epoch, step, end_of_epoch=False):
    p = CKPT_DIR / tag
    model.save_pretrained(p)
    (p / "training_state.json").write_text(
        json.dumps({"epoch": epoch, "step": step, "end_of_epoch": end_of_epoch})
    )

def find_latest_ckpt():
    epoch_ckpts = sorted(CKPT_DIR.glob("epoch_*"),
                         key=lambda p: int(p.name.split("_")[1]))
    candidates  = epoch_ckpts + ([CKPT_DIR / "latest"] if (CKPT_DIR / "latest").exists() else [])
    for p in reversed(candidates):
        if (p / "adapter_model.safetensors").exists():
            state = json.loads((p / "training_state.json").read_text()) if (p / "training_state.json").exists() else {}
            return p, state
    return None, {}

ckpt_path, ckpt_state = find_latest_ckpt()
start_epoch, step = 0, 0

if ckpt_path:
    print(f"Resuming from {ckpt_path.name} …")
    weights = sf.load_file(str(ckpt_path / "adapter_model.safetensors"), device="cpu")
    set_peft_model_state_dict(model, weights); del weights
    step = ckpt_state.get("step", 0)
    if ckpt_state.get("end_of_epoch", False):
        start_epoch = ckpt_state.get("epoch", 0) + 1
    else:
        start_epoch = ckpt_state.get("epoch", 0)
    print(f"  → resuming epoch {start_epoch + 1}, global step {step}")
else:
    print("No checkpoint — training from scratch.")

MAX_STEPS = 5 if SMOKE_TEST else (N_TRAIN * MAX_EPOCHS) // (TRAIN_BS * GRAD_ACCUM)
WARMUP    = max(1, MAX_STEPS // 10)

optimizer = AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=LR,
                  weight_decay=0.01)
scheduler = get_cosine_schedule_with_warmup(optimizer, WARMUP, MAX_STEPS)
if step > 0:
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", UserWarning)
        for _ in range(step):
            scheduler.step()

loader = DataLoader(train_cache, batch_size=TRAIN_BS, shuffle=True,
                    collate_fn=collate_train, num_workers=0)

autocast_ctx = torch.amp.autocast(device_type=DEVICE.type, dtype=DTYPE,
                                   enabled=DEVICE.type == "cuda")

model.train()
optimizer.zero_grad()
acc_loss     = 0.0
global_batch = 0
step_bar     = tqdm(total=MAX_STEPS, initial=step, desc="steps")

for epoch in range(start_epoch, 1 if SMOKE_TEST else MAX_EPOCHS):
    for batch in tqdm(loader, desc=f"epoch {epoch+1}", leave=False):
        fwd = {k: (v.to(DEVICE, dtype=DTYPE) if v.dtype.is_floating_point else v.to(DEVICE))
               for k, v in batch.items() if k != 'labels' and torch.is_tensor(v)}
        labels = batch['labels'].to(DEVICE)

        with autocast_ctx:
            loss = model(**fwd, labels=labels).loss / GRAD_ACCUM

        loss.backward()
        acc_loss     += loss.item()
        global_batch += 1

        if global_batch % GRAD_ACCUM == 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step(); scheduler.step(); optimizer.zero_grad()
            step += 1
            step_bar.update(1)
            step_bar.set_postfix(loss=f"{acc_loss:.4f}", ep=epoch + 1)
            acc_loss = 0.0
            if step % 50 == 0:
                save_ckpt("latest", epoch, step, end_of_epoch=False)
        if step >= MAX_STEPS:
            break

    save_ckpt(f"epoch_{epoch + 1}", epoch, step, end_of_epoch=True)
    print(f"\nCheckpoint saved → {CKPT_DIR / f'epoch_{epoch+1}'}")
    if step >= MAX_STEPS:
        break

step_bar.close()
model.eval()
print("Training complete.")

No checkpoint — training from scratch.


steps:   0%|          | 0/649 [00:00<?, ?it/s]

epoch 1:   0%|          | 0/520 [00:00<?, ?it/s]


Checkpoint saved → /content/drive/MyDrive/DL/endterm/checkpoints15/epoch_1


epoch 2:   0%|          | 0/520 [00:00<?, ?it/s]

...


Checkpoint saved → /content/drive/MyDrive/DL/endterm/checkpoints15/epoch_3


epoch 4:   0%|          | 0/520 [00:00<?, ?it/s]


Checkpoint saved → /content/drive/MyDrive/DL/endterm/checkpoints15/epoch_4


epoch 5:   0%|          | 0/520 [00:00<?, ?it/s]


Checkpoint saved → /content/drive/MyDrive/DL/endterm/checkpoints15/epoch_5
Training complete.


In [ ]:
# ── Optimised predict_losses(): image processed once per sample ────────────────

@torch.inference_mode()
def predict_losses(sample):
    """Returns list of length-normalised per-choice losses."""
    img       = sample['image']
    base_text = sample['text']
    n         = len(sample['choices'])

    base_enc   = processor(text=[base_text], images=[img], return_tensors='pt')
    prompt_len = base_enc['input_ids'].shape[1]
    cached_pv  = base_enc['pixel_values'].to(DEVICE, dtype=DTYPE)
    cached_pam = base_enc.get('pixel_attention_mask')
    if cached_pam is not None:
        cached_pam = cached_pam.to(DEVICE)

    base_ids  = base_enc['input_ids']
    base_amsk = base_enc['attention_mask']

    losses = []
    for i in range(n):
        ans_ids = processor.tokenizer.encode(
            f" {CHOICE_LETTERS[i]}", add_special_tokens=False, return_tensors='pt'
        )
        full_ids  = torch.cat([base_ids,  ans_ids],                  dim=1).to(DEVICE)
        full_amsk = torch.cat([base_amsk, torch.ones_like(ans_ids)], dim=1).to(DEVICE)

        labels = full_ids.clone()
        labels[:, :prompt_len] = -100
        labels[labels == processor.tokenizer.pad_token_id] = -100

        fwd = {'input_ids': full_ids, 'attention_mask': full_amsk,
               'pixel_values': cached_pv}
        if cached_pam is not None:
            fwd['pixel_attention_mask'] = cached_pam

        with torch.amp.autocast(device_type=DEVICE.type, dtype=DTYPE,
                                 enabled=DEVICE.type == 'cuda'):
            out = model(**fwd, labels=labels)

        n_toks = (labels != -100).sum().item()
        losses.append(out.loss.item() / max(n_toks, 1))
    return losses


# Mini-val accuracy check on attempt8's last checkpoint
correct = 0
for i in range(len(mini_val_ds)):
    pred = int(np.argmin(predict_losses(mini_val_ds[i])))
    if pred == mini_val_ds[i]['answer']:
        correct += 1
    if (i + 1) % max(1, len(mini_val_ds) // 5) == 0:
        print(f"  [{i+1}/{len(mini_val_ds)}] running acc: {correct/(i+1):.3f}")

print(f"\nMini-val accuracy: {correct}/{len(mini_val_ds)} = {correct/len(mini_val_ds):.4f}")

  [40/200] running acc: 0.975
  [80/200] running acc: 0.988
  [120/200] running acc: 0.983
  [160/200] running acc: 0.981
  [200/200] running acc: 0.965

Mini-val accuracy: 193/200 = 0.9650


In [ ]:
# ── Weight soup: average ep4 + ep5 from checkpoints15 ────────────────────────

import safetensors.torch as sf
from peft import set_peft_model_state_dict
from tqdm.auto import tqdm

base = Path('/content/drive/MyDrive/DL/endterm') if IN_COLAB else Path('.')

soup_ckpts = sorted(
    (base / 'checkpoints15').glob('epoch_*'),
    key=lambda p: int(p.name.split('_')[1])
)[-2:]
print(f"Souping: {[p.name for p in soup_ckpts]}")

soup_weights = None
for ckpt_p in soup_ckpts:
    w = sf.load_file(str(ckpt_p / 'adapter_model.safetensors'), device='cpu')
    if soup_weights is None:
        soup_weights = {k: v.clone() for k, v in w.items()}
    else:
        for k in soup_weights:
            soup_weights[k] += w[k]
    del w
for k in soup_weights:
    soup_weights[k] /= len(soup_ckpts)

set_peft_model_state_dict(model, soup_weights); del soup_weights
model.eval()
print(f"Soup of {len(soup_ckpts)} checkpoints loaded")

rows = []
for i in tqdm(range(len(test_ds)), desc='soup inference'):
    rows.append({'id': test_ds[i]['id'], 'answer': int(np.argmin(predict_losses(test_ds[i])))})

sub_soup = pd.DataFrame(rows)
sub_soup.to_csv('submission15_soup.csv', index=False)
print(f"Saved submission15_soup.csv ({len(sub_soup)} rows)")
if IN_COLAB:
    sub_soup.to_csv(base / 'submission15_soup.csv', index=False)
    print('Also saved to Drive')
print(sub_soup['answer'].value_counts().sort_index().to_string())
sub_soup.head()

Souping: ['epoch_4', 'epoch_5']
Soup of 2 checkpoints loaded


soup inference:   0%|          | 0/1008 [00:00<?, ?it/s]

Saved submission15_soup.csv (1008 rows)
Also saved to Drive
answer
0    364
1    347
2    225
3     72


,id,answer
0,test_01750,2
1,test_00128,0
2,test_02891,0
3,test_02425,1
4,test_00930,2


In [ ]:
# ── Single-model submission (attempt15 last checkpoint) ───────────────────────

rows = []
for i in range(len(test_ds)):

    rows.append({'id': test_ds[i]['id'], 'answer': int(np.argmin(predict_losses(test_ds[i])))})
    if (i + 1) % max(1, len(test_ds) // 5) == 0:
        print(f"  [{i+1}/{len(test_ds)}]")

sub = pd.DataFrame(rows)
sub.to_csv('submission15.csv', index=False)
print(f"Saved submission15.csv ({len(sub)} rows)")

if IN_COLAB:
    drive_path = Path('/content/drive/MyDrive/DL/endterm/submission15.csv')
    sub.to_csv(drive_path, index=False)
    print(f"Also saved to Drive: {drive_path}")

print(sub['answer'].value_counts().sort_index().to_string())
sub.head()

  [201/1008]
  [402/1008]
  [603/1008]
  [804/1008]
  [1005/1008]
Saved submission15.csv (1008 rows)
Also saved to Drive: /content/drive/MyDrive/DL/endterm/submission15.csv
answer
0    364
1    347
2    225
3     72


,id,answer
0,test_01750,2
1,test_00128,0
2,test_02891,0
3,test_02425,1
4,test_00930,2
